model.py


In [ ]:
%%writefile /content/drive/MyDrive/project_Landset08/training/model.py
import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------
# Conv Block
# -----------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

# -----------------------------
# Down block
# -----------------------------
class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = ConvBlock(in_ch, out_ch)

    def forward(self, x):
        return self.conv(self.pool(x))

# -----------------------------
# Up block
# -----------------------------
class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = ConvBlock(in_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)

        # Resize if mismatch due to odd dimensions
        if x.size() != skip.size():
            x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)

        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

# -----------------------------
# Fusion UNet Model
# -----------------------------
class UNetFusion(nn.Module):
    def __init__(self, base_filters=32):
        super().__init__()

        # 1 thermal + 3 RGB = 4 input channels
        in_channels = 4
        out_channels = 1

        self.inc = ConvBlock(in_channels, base_filters)
        self.down1 = Down(base_filters, base_filters*2)
        self.down2 = Down(base_filters*2, base_filters*4)
        self.down3 = Down(base_filters*4, base_filters*8)

        self.bottleneck = ConvBlock(base_filters*8, base_filters*16)

        self.up3 = Up(base_filters*16, base_filters*8)
        self.up2 = Up(base_filters*8*2, base_filters*4)
        self.up1 = Up(base_filters*4*2, base_filters*2)
        self.up0 = Up(base_filters*2*2, base_filters)

        self.out_conv = nn.Sequential(
            nn.Conv2d(base_filters*2, base_filters, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_filters, out_channels, 1)
        )

    def forward(self, lr_thermal, rgb):
        x = torch.cat([lr_thermal, rgb], dim=1)

        c0 = self.inc(x)
        c1 = self.down1(c0)
        c2 = self.down2(c1)
        c3 = self.down3(c2)

        b = self.bottleneck(c3)

        u3 = self.up3(b, c3)
        u2 = self.up2(u3, c2)
        u1 = self.up1(u2, c1)
        u0 = self.up0(u1, c0)

        out = self.out_conv(torch.cat([u0, c0], dim=1))
        return out


Overwriting /content/drive/MyDrive/project_Landset08/training/model.py


In [2]:
import os

TRAINING_FOLDER = "/content/drive/MyDrive/project_Landset08/training"
train_py_path = os.path.join(TRAINING_FOLDER, "train.py")

if os.path.exists(train_py_path):
    with open(train_py_path, 'r') as f:
        content = f.read()

    # Replace the incorrect path with the correct one
    # This is a robust replacement that will target the SPLIT_FILE line.
    updated_content = content.replace("splits/TRAIN.txt", "splits/train.txt")

    if updated_content != content:
        with open(train_py_path, 'w') as f:
            f.write(updated_content)
        print("✅ Corrected SPLIT_FILE path in train.py from TRAIN.txt to train.txt")
    else:
        print("No change needed for SPLIT_FILE path in train.py (already correct or TRAIN.txt not found).")
else:
    print(f"Error: train.py not found at {train_py_path}")

Error: train.py not found at /content/drive/MyDrive/project_Landset08/training/train.py


In [16]:
import os

TRAINING_FOLDER = "/content/drive/MyDrive/project_Landset08/training"
os.makedirs(TRAINING_FOLDER, exist_ok=True)

train_code = '''import os
import sys
import torch
from torch.utils.data import DataLoader
from torch import nn, optim
from tqdm import tqdm

# Add current directory to sys.path for module imports
# This is crucial when running with !python in a sub-process
script_dir = os.path.dirname(os.path.abspath(__file__))
if script_dir not in sys.path:
    sys.path.insert(0, script_dir)

# Debugging prints to check environment before imports
print(f"DEBUG: Current Working Directory INSIDE train.py: {os.getcwd()}")
print(f"DEBUG: sys.path INSIDE train.py: {sys.path}")

from dataset import ThermalDataset
from model import UNetFusion
from utils import ensure_dir

# ================================
# CONFIG
# ================================
BASE = "/content/drive/MyDrive/project_Landset08/final_dataset"

SPLIT_FILE = os.path.join(BASE, "splits/train.txt") # Ensure lowercase 'train'

BATCH_SIZE = 4
EPOCHS = 20
LR = 1e-4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ================================
# DATASET + LOADER
# ================================
dataset = ThermalDataset(
    split_file=SPLIT_FILE,
    base_dir=BASE # Pass base_dir as expected by dataset.py
)

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# ================================
# MODEL + LOSS + OPTIMIZER
# ================================
model = UNetFusion().to(DEVICE)
criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# ================================
# TRAIN LOOP
# ================================
for epoch in range(1, EPOCHS + 1):
    print()
    print(f"Epoch {epoch}/{EPOCHS}")
    loop = tqdm(loader)

    epoch_loss = 0.0

    for lr_img, rgb_img, hr_img in loop:
        lr_img = lr_img.to(DEVICE)
        rgb_img = rgb_img.to(DEVICE)
        hr_img = hr_img.to(DEVICE)

        optimizer.zero_grad()

        pred = model(lr_img, rgb_img)

        loss = criterion(pred, hr_img)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        loop.set_description(f"Loss: {loss.item():.6f}")

    print(f"Epoch {epoch} Average Loss: {epoch_loss/len(loader):.6f}")

# Save model
save_dir = "/content/drive/MyDrive/project_Landset08/training"
ensure_dir(save_dir)

save_path = os.path.join(save_dir, "sr_model.pth")
torch.save(model.state_dict(), save_path)

print("Model saved at:", save_path)
'''

with open(os.path.join(TRAINING_FOLDER, "train.py"), "w") as f:
    f.write(train_code)

print("✅ train.py re-created successfully with path and dataset args fixed!")

✅ train.py re-created successfully with path and dataset args fixed!


In [18]:
import os

BASE_DIR = "/content/drive/MyDrive/project_Landset08/final_dataset"
SPLITS_DIR = os.path.join(BASE_DIR, "splits")
TRAIN_SPLIT_FILE = os.path.join(SPLITS_DIR, "train.txt")

print(f"Checking for file: {TRAIN_SPLIT_FILE}")
if os.path.exists(TRAIN_SPLIT_FILE):
    print(f"✅ '{TRAIN_SPLIT_FILE}' exists!")
else:
    print(f"❌ '{TRAIN_SPLIT_FILE}' does NOT exist.")
    print(f"Listing contents of '{SPLITS_DIR}':")
    if os.path.exists(SPLITS_DIR) and os.path.isdir(SPLITS_DIR):
        for item in os.listdir(SPLITS_DIR):
            print(f"  - {item}")
    else:
        print(f"  '{SPLITS_DIR}' does not exist or is not a directory.")


Checking for file: /content/drive/MyDrive/project_Landset08/final_dataset/splits/train.txt
❌ '/content/drive/MyDrive/project_Landset08/final_dataset/splits/train.txt' does NOT exist.
Listing contents of '/content/drive/MyDrive/project_Landset08/final_dataset/splits':
  '/content/drive/MyDrive/project_Landset08/final_dataset/splits' does not exist or is not a directory.


In [14]:
%%writefile /content/drive/MyDrive/project_Landset08/training/model.py
import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------
# Conv Block
# -----------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

# -----------------------------
# Down block
# -----------------------------
class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = ConvBlock(in_ch, out_ch)

    def forward(self, x):
        return self.conv(self.pool(x))

# -----------------------------
# Up block
# -----------------------------
class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = ConvBlock(in_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)

        # Resize if mismatch due to odd dimensions
        if x.size() != skip.size():
            x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)

        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

# -----------------------------
# Fusion UNet Model
# -----------------------------
class UNetFusion(nn.Module):
    def __init__(self, base_filters=32):
        super().__init__()

        # 1 thermal + 3 RGB = 4 input channels
        in_channels = 4
        out_channels = 1

        self.inc = ConvBlock(in_channels, base_filters)
        self.down1 = Down(base_filters, base_filters*2)
        self.down2 = Down(base_filters*2, base_filters*4)
        self.down3 = Down(base_filters*4, base_filters*8)

        self.bottleneck = ConvBlock(base_filters*8, base_filters*16)

        self.up3 = Up(base_filters*16, base_filters*8)
        self.up2 = Up(base_filters*8*2, base_filters*4)
        self.up1 = Up(base_filters*4*2, base_filters*2)
        self.up0 = Up(base_filters*2*2, base_filters)

        self.out_conv = nn.Sequential(
            nn.Conv2d(base_filters*2, base_filters, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_filters, out_channels, 1)
        )

    def forward(self, lr_thermal, rgb):
        x = torch.cat([lr_thermal, rgb], dim=1)

        c0 = self.inc(x)
        c1 = self.down1(c0)
        c2 = self.down2(c1)
        c3 = self.down3(c2)

        b = self.bottleneck(c3)

        u3 = self.up3(b, c3)
        u2 = self.up2(u3, c2)
        u1 = self.up1(u2, c1)
        u0 = self.up0(u1, c0)

        out = self.out_conv(torch.cat([u0, c0], dim=1))
        return out

Writing /content/drive/MyDrive/project_Landset08/training/model.py


In [15]:
import os

TRAINING_FOLDER = "/content/drive/MyDrive/project_Landset08/training"

# Save current working directory
original_cwd = os.getcwd()

try:
    # Change current working directory to the training folder
    os.chdir(TRAINING_FOLDER)
    print(f"Changed current working directory to: {os.getcwd()}")

    # Now run the script. It should be able to find dataset.py and model.py directly.
    !python train.py
finally:
    # Change back to original directory
    os.chdir(original_cwd)
    print(f"Restored original working directory to: {os.getcwd()}")

Changed current working directory to: /content/drive/MyDrive/project_Landset08/training
Using device: cuda
Traceback (most recent call last):
  File "/content/drive/MyDrive/project_Landset08/training/train.py", line 35, in <module>
    dataset = ThermalDataset(
              ^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/project_Landset08/training/dataset.py", line 13, in __init__
    with open(split_file, "r") as f:
         ^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/project_Landset08/final_dataset/splits/train.txt'
Restored original working directory to: /content


In [11]:
import os

TRAINING_FOLDER = "/content/drive/MyDrive/project_Landset08/training"
os.makedirs(TRAINING_FOLDER, exist_ok=True)

utils_code = '''import os
import numpy as np
import torch

def ensure_dir(path):
    """Create directory if it does not exist."""
    if not os.path.exists(path):
        os.makedirs(path)

def minmax_norm(arr):
    """Normalize numpy array to [0,1]."""
    arr = arr.astype(np.float32)
    mn, mx = arr.min(), arr.max()
    if mx - mn < 1e-6:
        return arr
    return (arr - mn) / (mx - mn)

def tensor_to_numpy(t):
    """Convert tensor to numpy array."""
    return t.detach().cpu().numpy()
'''

with open(os.path.join(TRAINING_FOLDER, "utils.py"), "w") as f:
    f.write(utils_code)

print("✅ utils.py re-created successfully!")

✅ utils.py re-created successfully!


In [12]:
# ================================
# FINAL dataset.py (MATCHES YOUR DATASET)
# ================================

dataset_code = r'''
import os
import torch
from torch.utils.data import Dataset
import numpy as np

class ThermalDataset(Dataset):
    def __init__(self, split_file, base_dir):
        self.lr_dir  = os.path.join(base_dir, "LR_Thermal")
        self.rgb_dir = os.path.join(base_dir, "HR_Optical")
        self.hr_dir  = os.path.join(base_dir, "HR_Thermal")

        with open(split_file, "r") as f:
            ids = [line.strip() for line in f if line.strip()]

        self.valid_ids = []

        for name in ids:
            lr_path  = os.path.join(self.lr_dir,  f"{name}.npy")
            rgb_path = os.path.join(self.rgb_dir, f"{name}.npy")
            hr_path  = os.path.join(self.hr_dir,  f"{name}.npy")

            if os.path.exists(lr_path) and os.path.exists(rgb_path) and os.path.exists(hr_path):
                self.valid_ids.append(name)

        if len(self.valid_ids) == 0:
            raise RuntimeError("❌ No valid samples found. Folder names or paths are incorrect.")

        print(f"✅ Dataset loaded with {len(self.valid_ids)} valid samples")

    def __len__(self):
        return len(self.valid_ids)

    def __getitem__(self, idx):
        name = self.valid_ids[idx]

        lr  = np.load(os.path.join(self.lr_dir,  f"{name}.npy"))
        rgb = np.load(os.path.join(self.rgb_dir, f"{name}.npy"))
        hr  = np.load(os.path.join(self.hr_dir,  f"{name}.npy"))

        lr  = torch.from_numpy(lr).unsqueeze(0).float()   # (1,H,W)
        rgb = torch.from_numpy(rgb).permute(2, 0, 1).float()  # (3,H,W)
        hr  = torch.from_numpy(hr).unsqueeze(0).float()   # (1,H,W)

        return lr, rgb, hr
'''

path = os.path.join(TRAINING_FOLDER, "dataset.py")
with open(path, "w") as f:
    f.write(dataset_code)

print("✅ dataset.py recreated with correct folder structure and content!")

✅ dataset.py recreated with correct folder structure and content!


In [17]:
import os

TRAINING_FOLDER = "/content/drive/MyDrive/project_Landset08/training"

# Save current working directory
original_cwd = os.getcwd()

try:
    # Change current working directory to the training folder
    os.chdir(TRAINING_FOLDER)
    print(f"Changed current working directory to: {os.getcwd()}")

    # Now run the script. It should be able to find dataset.py and model.py directly.
    !python train.py
finally:
    # Change back to original directory
    os.chdir(original_cwd)
    print(f"Restored original working directory to: {os.getcwd()}")

Changed current working directory to: /content/drive/MyDrive/project_Landset08/training
DEBUG: Current Working Directory INSIDE train.py: /content/drive/MyDrive/project_Landset08/training
DEBUG: sys.path INSIDE train.py: ['/content/drive/MyDrive/project_Landset08/training', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages']
Using device: cuda
Traceback (most recent call last):
  File "/content/drive/MyDrive/project_Landset08/training/train.py", line 39, in <module>
    dataset = ThermalDataset(
              ^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/project_Landset08/training/dataset.py", line 13, in __init__
    with open(split_file, "r") as f:
         ^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/project_Landset08/final_dataset/splits/train.txt'
Restored original working directory to: /content


In [7]:
import os
import sys

TRAINING_FOLDER = "/content/drive/MyDrive/project_Landset08/training"

# Add your training folder to Python path if not already present
if TRAINING_FOLDER not in sys.path:
    sys.path.append(TRAINING_FOLDER)
    print(f"Added training folder to Python path: {TRAINING_FOLDER}")
else:
    print(f"Training folder already in Python path: {TRAINING_FOLDER}")


Added training folder to Python path: /content/drive/MyDrive/project_Landset08/training


In [8]:
!python /content/drive/MyDrive/project_Landset08/training/train.py

Traceback (most recent call last):
  File "/content/drive/MyDrive/project_Landset08/training/train.py", line 7, in <module>
    from dataset import ThermalDataset
ModuleNotFoundError: No module named 'dataset'


In [10]:
!python /content/drive/MyDrive/project_Landset08/training/train.py

Traceback (most recent call last):
  File "/content/drive/MyDrive/project_Landset08/training/train.py", line 14, in <module>
    from dataset import ThermalDataset
ModuleNotFoundError: No module named 'dataset'


In [3]:
!python /content/drive/MyDrive/project_Landset08/training/train.py

python3: can't open file '/content/drive/MyDrive/project_Landset08/training/train.py': [Errno 2] No such file or directory


DATASET.PY

In [ ]:
# ================================
# Auto-create dataset.py (SAFE VERSION)
# ================================

dataset_code = r'''
import os
import torch
from torch.utils.data import Dataset
import numpy as np

class ThermalDataset(Dataset):
    def __init__(self, split_file, lr_dir, rgb_dir, hr_dir):
        self.lr_dir = lr_dir
        self.rgb_dir = rgb_dir
        self.hr_dir = hr_dir

        # Read IDs from split file
        with open(split_file, "r") as f:
            ids = [line.strip() for line in f if line.strip()]

        self.valid_ids = []

        for name in ids:
            lr_path  = os.path.join(lr_dir,  f"{name}.npy")
            rgb_path = os.path.join(rgb_dir, f"{name}.npy")
            hr_path  = os.path.join(hr_dir,  f"{name}.npy")

            if os.path.exists(lr_path) and os.path.exists(rgb_path) and os.path.exists(hr_path):
                self.valid_ids.append(name)
            else:
                print(f"⚠️ Skipping missing sample: {name}")

        if len(self.valid_ids) == 0:
            raise RuntimeError("❌ No valid samples found. Check dataset paths.")

        print(f"✅ Dataset ready. Valid samples: {len(self.valid_ids)}")

    def __len__(self):
        return len(self.valid_ids)

    def __getitem__(self, idx):
        name = self.valid_ids[idx]

        lr  = np.load(os.path.join(self.lr_dir,  f"{name}.npy"))
        rgb = np.load(os.path.join(self.rgb_dir, f"{name}.npy"))
        hr  = np.load(os.path.join(self.hr_dir,  f"{name}.npy"))

        lr  = torch.from_numpy(lr).unsqueeze(0).float()
        rgb = torch.from_numpy(rgb).permute(2, 0, 1).float()
        hr  = torch.from_numpy(hr).unsqueeze(0).float()

        return lr, rgb, hr
'''

# Write dataset.py
save_path = "/content/drive/MyDrive/project_Landset08/training/dataset.py"
with open(save_path, "w") as f:
    f.write(dataset_code)

print("✅ dataset.py created successfully at:")
print(save_path)


✅ dataset.py created successfully at:
/content/drive/MyDrive/project_Landset08/training/dataset.py


TRAIN.PY

In [ ]:
import os

TARGET_DIR = "/content/drive/MyDrive/project_Landset08/training"
os.makedirs(TARGET_DIR, exist_ok=True)

train_path = os.path.join(TARGET_DIR, "train.py")

code = """
import os
import torch
from torch.utils.data import DataLoader
from torch import nn, optim
from tqdm import tqdm

from dataset import ThermalDataset
from model import UNetFusion
from utils import ensure_dir

# ================================
# CONFIG
# ================================
BASE = "/content/drive/MyDrive/project_Landset08/final_dataset"

SPLIT_FILE = os.path.join(BASE, "splits/TRAIN.txt")
LR_DIR     = os.path.join(BASE, "LR_Thermal")
RGB_DIR    = os.path.join(BASE, "HR_Optical")
HR_DIR     = os.path.join(BASE, "HR_Thermal")

BATCH_SIZE = 4
EPOCHS = 20
LR = 1e-4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ================================
# DATASET + LOADER
# ================================
dataset = ThermalDataset(
    split_file=SPLIT_FILE,
    base_dir=BASE
)


loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# ================================
# MODEL + LOSS + OPTIMIZER
# ================================
model = UNetFusion().to(DEVICE)
criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# ================================
# TRAIN LOOP
# ================================
for epoch in range(1, EPOCHS + 1):
    print()
    print(f"Epoch {epoch}/{EPOCHS}")

    loop = tqdm(loader)
    epoch_loss = 0.0

    for lr_img, rgb_img, hr_img in loop:
        lr_img  = lr_img.to(DEVICE)
        rgb_img = rgb_img.to(DEVICE)
        hr_img  = hr_img.to(DEVICE)

        optimizer.zero_grad()
        pred = model(lr_img, rgb_img)
        loss = criterion(pred, hr_img)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        loop.set_postfix(loss=f"{loss.item():.6f}")

    print("Epoch Average Loss:", epoch_loss / len(loader))

# ================================
# SAVE MODEL
# ================================
save_dir = "/content/drive/MyDrive/project_Landset08/training"
ensure_dir(save_dir)

save_path = os.path.join(save_dir, "sr_model.pth")
torch.save(model.state_dict(), save_path)

print("✅ Model saved at:", save_path)
"""

with open(train_path, "w") as f:
    f.write(code.strip())

print("✅ train.py recreated successfully at:")
print(train_path)


✅ train.py recreated successfully at:
/content/drive/MyDrive/project_Landset08/training/train.py


In [ ]:
import os

train_file = "/content/drive/MyDrive/project_Landset08/training/train.py"

with open(train_file, "r") as f:
    content = f.read()

content = content.replace("splits/TRAIN.txt", "splits/train.txt")

with open(train_file, "w") as f:
    f.write(content)

print("✅ train.py fixed to use splits/train.txt")


✅ train.py fixed to use splits/train.txt


utils.py

In [ ]:
# This cell previously wrote an outdated utils.py to /content/utils.py
# This file is causing an import conflict with the utils.py in the training folder.
# We will remove or comment out its content to prevent the error.
# %%writefile utils.py
# import numpy as np
# import torch

# def load_npy(path):
#     """Load a .npy file and return it as a torch tensor."""
#     try:
#         arr = np.load(path)
#     except Exception as e:
#         raise RuntimeError(f"Error loading file {path}: {e}")

#     # Convert to float32 tensor
#     tensor = torch.tensor(arr, dtype=torch.float32)

#     # Ensure channel-first format
#     if tensor.ndim == 2:
#         tensor = tensor.unsqueeze(0)         # (1, H, W)
#     elif tensor.ndim == 3:
#         tensor = tensor.permute(2, 0, 1)     # (C, H, W)
#     else:
#         raise ValueError(f"Invalid array shape in file {path}: {tensor.shape}")

#     return tensor


# def minmax_normalize(tensor):
#     """Normalize any tensor to range [0, 1]."""
#     min_val = tensor.min()
#     max_val = tensor.max()

#     if max_val - min_val == 0:
#         return tensor * 0  # Avoid division by zero

#     return (tensor - min_val) / (max_val - min_val)


# def print_tensor_info(name, tensor):
#     """Debug helper – prints shape, dtype, min, max."""
#     print(f"[INFO] {name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}, "
#           f"min={tensor.min().item():.4f}, max={tensor.max().item():.4f}")


# If you need to keep this content for other reasons, you should rename the file
# or explicitly ensure that `sys.path` prioritizes the training directory's utils.py

# Alternatively, you can explicitly remove the conflicting file if it was created:
import os
if os.path.exists('/content/utils.py'):
    os.remove('/content/utils.py')
    print("Removed conflicting /content/utils.py")


Overwriting utils.py


In [ ]:
!python /content/drive/MyDrive/project_Landset08/training/train.py


Using device: cuda

Epoch 1/20
  0% 0/116 [00:00<?, ?it/s]
Traceback (most recent call last):
  File "rasterio/_base.pyx", line 310, in rasterio._base.DatasetBase.__init__
  File "rasterio/_base.pyx", line 221, in rasterio._base.open_dataset
  File "rasterio/_err.pyx", line 359, in rasterio._err.exc_wrap_pointer
rasterio._err.CPLE_OpenFailedError: /content/drive/MyDrive/project_Landset08/final_dataset/HR_Thermal/0000148.tif: No such file or directory

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/drive/MyDrive/project_Landset08/training/train.py", line 38, in <module>
    for lr_img, rgb_img, hr_img in loop:
                                   ^^^^
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 1181, in __iter__
    for obj in iterable:
               ^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 732, in __next__
    data = self._next_data()
   

In [ ]:
import os

# ----------------------------
# Set your training folder path
# ----------------------------
TRAINING_FOLDER = "/content/drive/MyDrive/project_Landset08/training"
os.makedirs(TRAINING_FOLDER, exist_ok=True)

# ----------------------------
# Create utils.py
# ----------------------------
utils_code = '''import os
import numpy as np
import torch

def ensure_dir(path):
    """Create directory if it does not exist."""
    if not os.path.exists(path):
        os.makedirs(path)

def minmax_norm(arr):
    """Normalize numpy array to [0,1]."""
    arr = arr.astype(np.float32)
    mn, mx = arr.min(), arr.max()
    if mx - mn < 1e-6:
        return arr
    return (arr - mn) / (mx - mn)

def tensor_to_numpy(t):
    """Convert tensor to numpy array."""
    return t.detach().cpu().numpy()
'''

with open(os.path.join(TRAINING_FOLDER, "utils.py"), "w") as f:
    f.write(utils_code)

# ----------------------------
# Create train.py
# ----------------------------
train_code = '''import os
import torch
from torch.utils.data import DataLoader
from torch import nn, optim
from tqdm import tqdm

from dataset import ThermalDataset
from model import UNetFusion
from utils import ensure_dir

# ================================
# CONFIG
# ================================
BASE = "/content/drive/MyDrive/project_Landset08/final_dataset"

SPLIT_FILE = os.path.join(BASE, "splits/train.txt")
LR_DIR     = os.path.join(BASE, "LR_Thermal")  # Corrected folder name
RGB_DIR    = os.path.join(BASE, "HR_Optical") # Corrected folder name
HR_DIR     = os.path.join(BASE, "HR_Thermal") # Corrected folder name

BATCH_SIZE = 4
EPOCHS = 20
LR = 1e-4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ================================
# DATASET + LOADER
# ================================
dataset = ThermalDataset(
    split_file=SPLIT_FILE,
    lr_dir=LR_DIR,
    rgb_dir=RGB_DIR,
    hr_dir=HR_DIR
)

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# ================================
# MODEL + LOSS + OPTIMIZER
# ================================
model = UNetFusion().to(DEVICE)
criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# ================================
# TRAIN LOOP
# ================================
for epoch in range(1, EPOCHS + 1):
    print()
    print(f"Epoch {epoch}/{EPOCHS}")
    loop = tqdm(loader)

    epoch_loss = 0.0

    for lr_img, rgb_img, hr_img in loop:
        lr_img = lr_img.to(DEVICE)
        rgb_img = rgb_img.to(DEVICE)
        hr_img = hr_img.to(DEVICE)

        optimizer.zero_grad()

        pred = model(lr_img, rgb_img)

        loss = criterion(pred, hr_img)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        loop.set_description(f"Loss: {loss.item():.6f}")

    print(f"Epoch {epoch} Average Loss: {epoch_loss/len(loader):.6f}")

# Save model
save_dir = "/content/drive/MyDrive/project_Landset08/training"
ensure_dir(save_dir)

save_path = os.path.join(save_dir, "sr_model.pth")
torch.save(model.state_dict(), save_path)

print("Model saved at:", save_path)
'''

with open(os.path.join(TRAINING_FOLDER, "train.py"), "w") as f:
    f.write(train_code)

print("✅ utils.py and train.py created/overwritten successfully!")

✅ utils.py and train.py created/overwritten successfully!


In [ ]:
import os

TRAINING_FOLDER = "/content/drive/MyDrive/project_Landset08/training"
train_py_path = os.path.join(TRAINING_FOLDER, "train.py")

if os.path.exists(train_py_path):
    with open(train_py_path, 'r') as f:
        lines = f.readlines()

    # The error was detected at line 51
    line_number_in_error = 51

    if 0 < line_number_in_error <= len(lines):
        print(f"Content of line {line_number_in_error} in train.py:\n{lines[line_number_in_error - 1].strip()}")
    else:
        print(f"Line {line_number_in_error} is out of bounds for train.py which has {len(lines)} lines.")
else:
    print(f"File not found: {train_py_path}")

Content of line 51 in train.py:
print(f"


check/ testing snippet

Training

In [ ]:
import os
import sys

# -----------------------------
# Remove conflicting utils.py
# -----------------------------
conflict_file = "/content/utils.py"
if os.path.exists(conflict_file):
    os.remove(conflict_file)
    print(f"Removed conflicting file: {conflict_file}")
else:
    print(f"No conflicting utils.py in /content/")

# -----------------------------
# Add your training folder to Python path
# -----------------------------
TRAINING_FOLDER = "/content/drive/MyDrive/project_Landset08/training"
if TRAINING_FOLDER not in sys.path:
    sys.path.append(TRAINING_FOLDER)
    print(f"Added training folder to Python path: {TRAINING_FOLDER}")

# -----------------------------
# Test imports
# -----------------------------
try:
    from utils import ensure_dir
    from model import UNetFusion
    from dataset import ThermalDataset
    print("✅ Imports successful. All files are using the correct paths.")
except Exception as e:
    print("❌ Import failed:", e)

# -----------------------------
# Optional: check utils.py function
# -----------------------------
# This line was causing a NameError if ensure_dir was not imported,
# so commenting it out for now to allow the cell to run without error
# print("ensure_dir function exists:", callable(ensure_dir))


No conflicting utils.py in /content/
Added training folder to Python path: /content/drive/MyDrive/project_Landset08/training
✅ Imports successful. All files are using the correct paths.


In [ ]:
import os
os.kill(os.getpid(), 9)


In [ ]:
split_file = "/content/drive/MyDrive/project_Landset08/final_dataset/splits/train.txt"

with open(split_file) as f:
    lines = f.readlines()

print("Total image IDs:", len(lines))
print("First 5 IDs:", lines[:5])


Total image IDs: 462
First 5 IDs: ['0000569\n', '0000088\n', '0000514\n', '0000048\n', '0000497\n']


training 01

In [ ]:
import os

base = "/content/drive/MyDrive/project_Landset08/training"
print(os.listdir(base))


['__pycache__', 'checkpoints', 'logs', 'model.py', 'utils.py', 'train.py', 'dataset.py']


In [ ]:
# ================================
# FINAL dataset.py (MATCHES YOUR DATASET)
# ================================

dataset_code = r'''
import os
import torch
from torch.utils.data import Dataset
import numpy as np

class ThermalDataset(Dataset):
    def __init__(self, split_file, base_dir):
        self.lr_dir  = os.path.join(base_dir, "LR_Thermal")
        self.rgb_dir = os.path.join(base_dir, "HR_Optical")
        self.hr_dir  = os.path.join(base_dir, "HR_Thermal")

        with open(split_file, "r") as f:
            ids = [line.strip() for line in f if line.strip()]

        self.valid_ids = []

        for name in ids:
            lr_path  = os.path.join(self.lr_dir,  f"{name}.npy")
            rgb_path = os.path.join(self.rgb_dir, f"{name}.npy")
            hr_path  = os.path.join(self.hr_dir,  f"{name}.npy")

            if os.path.exists(lr_path) and os.path.exists(rgb_path) and os.path.exists(hr_path):
                self.valid_ids.append(name)

        if len(self.valid_ids) == 0:
            raise RuntimeError("❌ No valid samples found. Folder names or paths are incorrect.")

        print(f"✅ Dataset loaded with {len(self.valid_ids)} valid samples")

    def __len__(self):
        return len(self.valid_ids)

    def __getitem__(self, idx):
        name = self.valid_ids[idx]

        lr  = np.load(os.path.join(self.lr_dir,  f"{name}.npy"))
        rgb = np.load(os.path.join(self.rgb_dir, f"{name}.npy"))
        hr  = np.load(os.path.join(self.hr_dir,  f"{name}.npy"))

        lr  = torch.from_numpy(lr).unsqueeze(0).float()   # (1,H,W)
        rgb = torch.from_numpy(rgb).permute(2, 0, 1).float()  # (3,H,W)
        hr  = torch.from_numpy(hr).unsqueeze(0).float()   # (1,H,W)

        return lr, rgb, hr
'''

path = "/content/drive/MyDrive/project_Landset08/training/dataset.py"
with open(path, "w") as f:
    f.write(dataset_code)

print("✅ dataset.py recreated with correct folder structure")


✅ dataset.py recreated with correct folder structure


In [ ]:
!python /content/drive/MyDrive/project_Landset08/training/train.py


Using device: cuda
Traceback (most recent call last):
  File "/content/drive/MyDrive/project_Landset08/training/train.py", line 31, in <module>
    dataset = ThermalDataset(
              ^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/project_Landset08/training/dataset.py", line 13, in __init__
    with open(split_file, "r") as f:
         ^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/project_Landset08/final_dataset/splits/TRAIN.txt'
